# Experiment 3: Layered Failure Analysis

L1-L5 failure distribution, layer-conditioned fragility, CF-exposed failures.

**Core thesis claim**: AL failures concentrate in L3 (Event-driven), L4 (Workflow), L5 (Toolchain).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# These modules will be available once counterfactual evaluation is implemented on top of main.
# TODO: Import from bcbench.analysis.family, bcbench.types, and evaluator.thesis_metrics once available
#   from bcbench.analysis.family import FamilyOutcome
#   from bcbench.types import FailureLayer
#   from evaluator.thesis_metrics import (
#       cf_exposed_failure_count, failure_layer_distribution, layer_conditioned_fragility,
#   )

LAYER_ORDER = [
    "L1-syntax-representation", "L2-execution-validation",
    "L3-event-driven-paradigm", "L4-workflow-business-logic",
    "L5-toolchain-ecosystem",
]
LAYER_SHORT = {"L1-syntax-representation": "L1", "L2-execution-validation": "L2",
               "L3-event-driven-paradigm": "L3", "L4-workflow-business-logic": "L4",
               "L5-toolchain-ecosystem": "L5"}

print("Imports ready. Load families from experiment results to begin analysis.")

## L1-L5 Failure Distribution

In [ ]:
from collections import Counter


def fragility_rate(families: list) -> float:
    eligible = [f for f in families if f.base.passed]
    if not eligible:
        return 0.0
    return sum(1 for f in eligible if f.is_fragile) / len(eligible)


def failure_layer_distribution(families: list) -> dict[str, int]:
    return dict(sorted(Counter(f.failure_layer.value for f in families if f.failure_layer is not None).items()))


def layer_conditioned_fragility(families: list) -> dict[str, float]:
    by_layer: dict[str, list] = {}
    for f in families:
        if f.failure_layer is None:
            continue
        by_layer.setdefault(f.failure_layer.value, []).append(f)
    return {layer: fragility_rate(fams) for layer, fams in sorted(by_layer.items())}


def cf_exposed_failure_count(families: list) -> int:
    return sum(1 for f in families if f.is_fragile and f.cf_fail_count == f.cf_total)


def plot_failure_layer_distribution(families_by_model: dict[str, list]):
    rows = []
    for model, families in families_by_model.items():
        dist = failure_layer_distribution(families)
        total = sum(dist.values()) or 1
        for layer in LAYER_ORDER:
            count = dist.get(layer, 0)
            rows.append({
                "Model": model,
                "Layer": LAYER_SHORT.get(layer, layer),
                "Count": count,
                "Proportion (%)": round(count / total * 100, 1),
            })

    df = pd.DataFrame(rows)
    fig = px.bar(
        df, x="Layer", y="Proportion (%)", color="Model",
        barmode="group", title="Failure Layer Distribution (L1-L5) by Model",
        text_auto=True,
        category_orders={"Layer": ["L1", "L2", "L3", "L4", "L5"]},
    )
    fig.show()
    return df


print("plot_failure_layer_distribution() ready")

## Layer-Conditioned Fragility Heatmap

Fragile family % per layer per model — the core diagnostic matrix.

In [ ]:
def plot_layer_conditioned_fragility(families_by_model: dict[str, list]):
    models = list(families_by_model.keys())
    layers = ["L1", "L2", "L3", "L4", "L5"]
    matrix = []

    for model in models:
        lcf = layer_conditioned_fragility(families_by_model[model])
        row = [round(lcf.get(layer, 0) * 100, 1) for layer in LAYER_ORDER]
        matrix.append(row)

    fig = go.Figure(data=go.Heatmap(
        z=matrix, x=layers, y=models,
        colorscale="RdYlGn_r", text=matrix, texttemplate="%{text}%",
        colorbar_title="Fragility %",
    ))
    fig.update_layout(
        title="Layer-Conditioned Fragility: P(CF fail | Base correct) per Layer",
        xaxis_title="Failure Layer", yaxis_title="Model",
    )
    fig.show()


print("plot_layer_conditioned_fragility() ready")

## Layer-Conditioned Severity

In [ ]:
def plot_severity_by_layer(families: list):
    rows = []
    for f in families:
        if f.failure_layer and f.severity is not None:
            rows.append({
                "Layer": LAYER_SHORT.get(f.failure_layer.value, f.failure_layer.value),
                "Severity": f.severity,
                "Type": f.family_type.value,
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        fig = px.box(
            df, x="Layer", y="Severity", color="Layer",
            title="Fragility Severity by Failure Layer",
            points="all",
            category_orders={"Layer": ["L1", "L2", "L3", "L4", "L5"]},
        )
        fig.show()
    return df


print("plot_severity_by_layer() ready")

## CF-Exposed Failure Analysis

Families where failure ONLY appears in CF (base correct, all CFs fail).

In [ ]:
def analyze_cf_exposed(families_by_model: dict[str, list]):
    rows = []
    for model, families in families_by_model.items():
        total = len(families)
        exposed = cf_exposed_failure_count(families)
        fragile = sum(1 for f in families if f.is_fragile)
        rows.append({
            "Model": model,
            "Total Families": total,
            "Fragile": fragile,
            "CF-Exposed (all CFs fail)": exposed,
            "CF-Exposed %": round(exposed / total * 100, 1) if total else 0,
        })

    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    return df


print("analyze_cf_exposed() ready")